In [ ]:
"""
==============================================================================
 COMPARACIÓN ARIMA / SARIMA / SARIMAX por centro
==============================================================================

   - ARIMA   : usa solo el pasado de la serie (autoregresión + media móvil)
   - SARIMA  : ARIMA + componente estacional
   - SARIMAX : SARIMA + variables físicas exógenas (temperatura, salinidad,
               capa de mezcla, altura del mar, velocidad de corriente),
               coherente con las variables usadas en RF y XGBoost.
 BLOQUES:
   1 Configuración   2 Carga por centro   3 Ajuste de los 3 modelos
   4 Evaluación y reporte   5 Main
==============================================================================
"""

import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"  : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"    : "Total_Parasitos_",
    "exogenas"  : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", #"Velocidad_Corriente_m_s",
    ],
    "col_centro": "id_centro",
    "col_fecha" : "Fecha",
    "test_frac" : 0.40,          # último 20% como test temporal
    "orden_arima"      : (2, 1, 2),    
    "orden_estacional" : (1, 0, 1, 14),     
}


# ============================================================================
# BLOQUE 2 — CARGA DE LA SERIE DE UN CENTRO
# ============================================================================
def cargar_serie_centro(config, centro):
    df = pd.read_csv(config["ruta_csv"])
    df[config["col_fecha"]] = pd.to_datetime(df[config["col_fecha"]])
    df = df[df[config["target"]].notna()]
    sub = df[df[config["col_centro"]] == centro].sort_values(config["col_fecha"])
    sub = sub.set_index(config["col_fecha"])

    
    y = sub[config["target"]].astype(float)
    Xex = sub[config["exogenas"]].astype(float)

    
    idx = pd.date_range(y.index.min(), y.index.max(), freq="D")
    y = y.reindex(idx).interpolate(method="time").fillna(method="bfill")
    Xex = Xex.reindex(idx).interpolate(method="time").fillna(method="bfill")
    return y, Xex


# ============================================================================
# BLOQUE 3 — AJUSTE DE LOS TRES MODELOS
# ============================================================================
def dividir_train_test(y, Xex, test_frac):
    n = len(y)
    corte = int(n * (1 - test_frac))
    return (y.iloc[:corte], y.iloc[corte:],
            Xex.iloc[:corte], Xex.iloc[corte:])


def ajustar_arima(ytr, yte, config):
    modelo = ARIMA(ytr, order=config["orden_arima"]).fit()
    pred = modelo.forecast(steps=len(yte))
    return np.asarray(pred)


def ajustar_sarima(ytr, yte, config):
    modelo = SARIMAX(ytr, order=config["orden_arima"],
                     seasonal_order=config["orden_estacional"],
                     enforce_stationarity=False,
                     enforce_invertibility=False).fit(disp=False)
    pred = modelo.forecast(steps=len(yte))
    return np.asarray(pred)


def ajustar_sarimax(ytr, yte, Xtr, Xte, config):
    modelo = SARIMAX(ytr, exog=Xtr, order=config["orden_arima"],
                     seasonal_order=config["orden_estacional"],
                     enforce_stationarity=False,
                     enforce_invertibility=False).fit(disp=False)
    pred = modelo.forecast(steps=len(yte), exog=Xte)
    return np.asarray(pred)


# ============================================================================
# BLOQUE 4 — EVALUACIÓN Y REPORTE
# ============================================================================
def metricas(y_true, y_pred):
    y_true = np.asarray(y_true)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {
        "MAE" : mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2"  : r2_score(y_true, y_pred),
    }


def evaluar_centro(config, centro):
    print("=" * 70)
    print(f"  CENTRO {centro}")
    print("=" * 70)
    y, Xex = cargar_serie_centro(config, centro)
    ytr, yte, Xtr, Xte = dividir_train_test(y, Xex, config["test_frac"])
    print(f"  Serie diaria: {len(y)} días | train={len(ytr)} test={len(yte)}")

    resultados = {}
    # ARIMA
    try:
        pred = ajustar_arima(ytr, yte, config)
        resultados["ARIMA"] = metricas(yte, pred)
    except Exception as e:
        print(f"  ARIMA falló: {e}")
    # SARIMA
    try:
        pred = ajustar_sarima(ytr, yte, config)
        resultados["SARIMA"] = metricas(yte, pred)
    except Exception as e:
        print(f"  SARIMA falló: {e}")
    # SARIMAX
    try:
        pred = ajustar_sarimax(ytr, yte, Xtr, Xte, config)
        resultados["SARIMAX"] = metricas(yte, pred)
    except Exception as e:
        print(f"  SARIMAX falló: {e}")

    # Tabla comparativa
    print(f"\n  {'Modelo':<10}{'MAE':<10}{'RMSE':<10}{'R2':<10}")
    print("  " + "-" * 40)
    mejor = None
    for nombre, m in resultados.items():
        print(f"  {nombre:<10}{m['MAE']:<10.4f}{m['RMSE']:<10.4f}{m['R2']:<10.4f}")
        if mejor is None or m["MAE"] < resultados[mejor]["MAE"]:
            mejor = nombre
    if mejor:
        print(f"\n  → Mejor por MAE: {mejor}")
    print()
    return resultados


# ============================================================================
# BLOQUE 5 — MAIN
# ============================================================================
def main():
    df = pd.read_csv(CONFIG["ruta_csv"])
    centros = df[df[CONFIG["target"]].notna()][CONFIG["col_centro"]].unique()
    print("\nComparación ARIMA / SARIMA / SARIMAX por centro\n")
    for centro in centros:
        evaluar_centro(CONFIG, centro)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 TEST DE ESTACIONARIEDAD POR VARIABLE — ADF + KPSS
==============================================================================

==============================================================================
"""

import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
warnings.filterwarnings("ignore")

CONFIG = {
    "ruta_csv"  : "data/Dataset_Maestro_Amplificado_ML.csv",
    "col_centro": "id_centro",
    "col_fecha" : "Fecha",
    # Todas las variables a analizar: parasitaria + físicas
    "variables" : [
        "Total_Parasitos_",
        "Hembras_Ovigeras",
        "Adultos_Moviles",
        "Juveniles",
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
}


def diagnostico_combinado(p_adf, p_kpss):
    """Combina ADF (H0: no estac.) y KPSS (H0: estac.) en un veredicto."""
    adf_estac = p_adf < 0.05
    kpss_estac = p_kpss >= 0.05
    if adf_estac and kpss_estac:
        return "ESTACIONARIA"
    elif not adf_estac and not kpss_estac:
        return "NO estacionaria"
    elif adf_estac and not kpss_estac:
        return "Tendencia-estac. (ambiguo)"
    else:
        return "Raíz unitaria débil (ambiguo)"


def test_variable(serie):
    serie = serie[~np.isnan(serie)]
    # ADF
    try:
        p_adf = adfuller(serie, autolag="AIC")[1]
    except Exception:
        p_adf = np.nan
    # KPSS
    try:
        p_kpss = kpss(serie, regression="c", nlags="auto")[1]
    except Exception:
        p_kpss = np.nan
    veredicto = diagnostico_combinado(p_adf, p_kpss)
    return p_adf, p_kpss, veredicto


def main():
    df = pd.read_csv(CONFIG["ruta_csv"])
    df[CONFIG["col_fecha"]] = pd.to_datetime(df[CONFIG["col_fecha"]])

    for centro in df[CONFIG["col_centro"]].unique():
        sub = df[df[CONFIG["col_centro"]] == centro].sort_values(CONFIG["col_fecha"])
        print("=" * 78)
        print(f"  CENTRO {centro}  (n={len(sub)})")
        print("=" * 78)
        print(f"  {'Variable':<26}{'p-ADF':<10}{'p-KPSS':<10}{'Veredicto'}")
        print("  " + "-" * 74)
        for var in CONFIG["variables"]:
            if var not in sub.columns:
                continue
            serie = sub[var].astype(float).values
            
            if np.isnan(serie).all() or len(serie[~np.isnan(serie)]) < 20:
                continue
            p_adf, p_kpss, veredicto = test_variable(serie)
            padf_s = f"{p_adf:.4f}" if not np.isnan(p_adf) else "N/A"
            pkpss_s = f"{p_kpss:.4f}" if not np.isnan(p_kpss) else "N/A"
            print(f"  {var:<26}{padf_s:<10}{pkpss_s:<10}{veredicto}")
        print()


if __name__ == "__main__":
    main()